Importing required libraries

In [1]:

import sqlite3
import datetime
import re
from typing import Tuple, List, Any, Optional
import pandas as pd



Creating Database and Assistant Class

In [2]:

class DatabaseChatAssistant:
    def __init__(self, db_path: str):
        """Initialize the chat assistant with database connection."""
        self.db_path = db_path
        self.create_database()
        
    def create_database(self) -> None:
        """Create the database and populate with initial data."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Create tables
        cursor.executescript('''
            CREATE TABLE IF NOT EXISTS Employees (
                ID INTEGER PRIMARY KEY,
                Name TEXT NOT NULL,
                Department TEXT NOT NULL,
                Salary INTEGER NOT NULL,
                Hire_Date DATE NOT NULL
            );
            
            CREATE TABLE IF NOT EXISTS Departments (
                ID INTEGER PRIMARY KEY,
                Name TEXT NOT NULL,
                Manager TEXT NOT NULL
            );
        ''')
        
        # Insert initial data if tables are empty
        cursor.execute("SELECT COUNT(*) FROM Employees")
        if cursor.fetchone()[0] == 0:
            cursor.executescript('''
                INSERT INTO Employees VALUES
                    (1, 'Alice', 'Sales', 50000, '2021-01-15'),
                    (2, 'Bob', 'Engineering', 70000, '2020-06-10'),
                    (3, 'Charlie', 'Marketing', 60000, '2022-03-20'),
                    (4, 'David', 'Sales', 55000, '2021-08-01'),
                    (5, 'Eve', 'Engineering', 75000, '2020-03-15'),
                    (6, 'Frank', 'Marketing', 62000, '2022-06-10');
                    
                INSERT INTO Departments VALUES
                    (1, 'Sales', 'Alice'),
                    (2, 'Engineering', 'Bob'),
                    (3, 'Marketing', 'Charlie');
            ''')
            
        conn.commit()
        conn.close()
    
    def parse_query(self, query: str) -> Tuple[str, dict]:
        """Parse natural language query and return SQL query with parameters."""
        query = query.lower().strip()
        
        # Pattern matching for different query types
        if "all employees in" in query or "show me all employees in" in query:
            department = re.search(r'in the (\w+) department', query)
            if department:
                return (
                    "SELECT * FROM Employees WHERE LOWER(Department) = LOWER(?)",
                    {"params": (department.group(1),)}
                )
                
        elif "manager of" in query:
            department = re.search(r'manager of the (\w+) department', query)
            if department:
                return (
                    "SELECT Manager FROM Departments WHERE LOWER(Name) = LOWER(?)",
                    {"params": (department.group(1),)}
                )
                
        elif "hired after" in query:
            date = re.search(r'hired after (\d{4}-\d{2}-\d{2})', query)
            if date:
                return (
                    "SELECT * FROM Employees WHERE Hire_Date > ?",
                    {"params": (date.group(1),)}
                )
                
        elif "total salary" in query:
            department = re.search(r'for the (\w+) department', query)
            if department:
                return (
                    "SELECT SUM(Salary) as TotalSalary FROM Employees WHERE LOWER(Department) = LOWER(?)",
                    {"params": (department.group(1),)}
                )
        
        raise ValueError("I couldn't understand that query. Please try rephrasing it.")
    
    def execute_query(self, sql: str, params: dict) -> pd.DataFrame:
        """Execute SQL query and return results as a pandas DataFrame."""
        try:
            conn = sqlite3.connect(self.db_path)
            df = pd.read_sql_query(sql, conn, params=params["params"])
            conn.close()
            return df
        except sqlite3.Error as e:
            raise Exception(f"Database error: {str(e)}")
    
    def format_response(self, query: str, df: pd.DataFrame) -> str:
        """Format query results into a natural language response."""
        if df.empty:
            return "I couldn't find any results matching your query."
            
        if "all employees in" in query.lower():
            response = "Here are the employees in that department:\n\n"
            for _, row in df.iterrows():
                response += f"- {row['Name']}: Hired on {row['Hire_Date']}, Salary: ${row['Salary']:,}\n"
            return response
            
        elif "manager of" in query.lower():
            return f"The manager of that department is {df['Manager'].iloc[0]}."
            
        elif "hired after" in query.lower():
            response = "Here are the employees hired after that date:\n\n"
            for _, row in df.iterrows():
                response += f"- {row['Name']} ({row['Department']} department): Hired on {row['Hire_Date']}\n"
            return response
            
        elif "total salary" in query.lower():
            total = df['TotalSalary'].iloc[0]
            return f"The total salary expense for that department is ${total:,}"
            
        return df.to_string()
    
    def process_query(self, query: str) -> str:
        """Main method to process a natural language query and return a response."""
        try:
            sql, params = self.parse_query(query)
            results_df = self.execute_query(sql, params)
            return self.format_response(query, results_df)
        except ValueError as e:
            return str(e)
        except Exception as e:
            return f"An error occurred: {str(e)}"

# Create an instance of the chat assistant
assistant = DatabaseChatAssistant('company.db')



Interactive query cell

In [5]:

def run_query():
    query = input("Enter your query (or 'quit' to exit): ")
    if query.lower() != 'quit':
        print("\nResponse:")
        print(assistant.process_query(query))
        return True
    return False

# Run this cell multiple times to test different queries
run_query()


Response:
The total salary expense for that department is $122,000


True